# ContractRisk — continuación con evaluador de evidencia 1.1

Corrección del evaluador tras inspeccionar el piloto 05: permite usar un fragmento disponible no citado por la respuesta SOLO como diagnóstico si se declara contradicción/incompletitud y citas sin respaldo. No convierte ese fragmento en cita de la respuesta. Conserva controles de literalidad, fuente conocida, rúbrica y umbrales.

No cambia el modelo, prompt, muestreo ni las siete notas originales. Verifica sus hashes, recalcula con la regla nueva y reutiliza todas. Ejecuta primero los dos casos pendientes del piloto; solo si pasan continúa con los 29 juicios restantes (31 llamadas nuevas en total si completa todo, más posibles reintentos de formato). El resultado combinado se identifica como dos sesiones, no una corrida nueva de 38.

Corrección post-hoc, no validación ciega. Los siete casos previos no se regeneran ni se seleccionan. La ausencia de error automático no demuestra buen juicio semántico; auditoría humana y confirmación nueva siguen pendientes.

Sesión nueva T4 y acceso a WANDB_API_KEY. Se requiere el mismo perfil de paquetes/GPU que la sesión original; si difiere, se detiene antes de cargar pesos para decidir cómo proceder sin mezclar resultados silenciosamente. No modificar versiones por ensayo y error. Sin cambios en main ni Space.


In [ ]:
import os, sys, subprocess, tempfile, json
from pathlib import Path

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_final_literal_continuation_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', '6ec8dcdf22433c632b5eb0e5539401c33b9c923d'], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError(f'Proceso terminó con código {status}; consultar {logfile}')

from google.colab import userdata
try:
    key = userdata.get('WANDB_API_KEY')
    if not key:
        raise ValueError('Empty secret')
    os.environ['WANDB_API_KEY'] = key
    del key
except Exception:
    raise RuntimeError('Habilita el acceso de ESTE notebook a WANDB_API_KEY en el panel de secretos de Colab. No pegues ni imprimas la clave.') from None
print('Secreto accesible; su valor no se imprime. La conexión se validará antes de cargar modelos.')


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r',
    'final/requirements-calibration.txt'], 'installation.log')
import torch
assert torch.cuda.is_available(), 'Selecciona GPU T4 y una sesión nueva.'
print('GPU:', torch.cuda.get_device_name(0))
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('20 anclas aprobadas; hashes y rúbrica comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
RESULTS = ROOT / 'literal_continuation_results'
run_visible([sys.executable, '-u', 'final/run_literal_continuation.py', '--output', str(RESULTS), '--require-wandb'], 'literal_continuation.log')


## Conservación e interpretación

Un piloto detenido puede contener las siete notas reutilizadas y una o dos nuevas; metadata distingue judgements_executed de judgements_total. Nunca interpretar los casos no ejecutados como aprobados. No repetir para escoger puntuaciones favorables. Si falla el perfil de entorno, se conserva el diagnóstico y no se mezclan sesiones.


In [ ]:
RESULTS = ROOT / 'literal_continuation_results'
for name in ['metadata.json', 'pilot_summary.json', 'regression_summary.json']:
    path = RESULTS / name
    if path.exists():
        print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, RESULTS / log.name)
reviewed = RESULTS / 'reviewed_calibration_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/calibration_review_v1', reviewed)
shutil.copy2(ROOT / 'final/judge_continuation_config.json', RESULTS / 'judge_continuation_config.json')
package = shutil.make_archive(str(ROOT / 'contractrisk_final_juez_continuacion_resultados'), 'zip', RESULTS)
files.download(package)
